In [6]:
from langchain.agents import create_agent
from IPython.display import Markdown
from dotenv.ipython import load_dotenv

In [8]:
load_dotenv(override=True)

True

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver 

In [27]:
agent = create_agent(
    model="openai:gpt-4o", 
    tools=[],
    system_prompt= "You are a helpful assistant", 
    )


In [26]:
response = agent.invoke(
    input={
        'messages':[
              {'role':'user', 'content':'My name is Mohamed'}
             ]
            })

In [10]:
print(display(Markdown(response['messages'][-1].content)))

Hello, Mohamed! How can I assist you today?

None


In [31]:
response = agent.invoke(
    input={
        'messages':[
              {'role':'user', 'content':'What is My Name'}
             ]
            }
            )

In [32]:
print(display(Markdown(response['messages'][-1].content)))

I'm sorry, I don't have the ability to know your name. But if you'd like, you can tell me!

None


In [58]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()
agent = create_agent(
    model="openai:gpt-4o", 
    tools=[],
    system_prompt= "You are a helpful assistant", 
    checkpointer=memory
    )

In [51]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage

In [44]:
config = {"configurable": {"thread_id": "1"}}
resp=agent.invoke(
    input={"messages":[HumanMessage(content="je m'appelle mohammed?")]},
    config=config
)


In [46]:
print(display(Markdown(resp['messages'][-1].content)))

D'accord, vous vous appelez Mohammed. Que puis-je faire pour vous aujourd'hui ?

None


In [47]:
config = {"configurable": {"thread_id": "1"}}
resp=agent.invoke(
    input={"messages":[HumanMessage(content="comment je m'appelle?")]},
    config=config
)

In [48]:
print(display(Markdown(resp['messages'][-1].content)))

Vous m'avez dit que votre nom est Mohammed. Comment puis-je vous aider aujourd'hui ?

None


In [52]:
from langchain.tools import tool

In [56]:
@tool
def get_weather(city: str):
    """
    Get the current weather for a given location.
    """
    print("Weather Tool invoked")
    return {
        "city": city,
        "temperature": "20°C",
        "description": "Partly cloudy"
    }

@tool
def get_employee_info(employee_name: str):
    """
    Get information about a specific employee(salary, seniority)
    """
    print("get_employee_info Tool invoked")
    return {
        "name": employee_name,
        "position": "Software Engineer",
        "department": "Engineering",
        "salary": "$100,000",
        "seniority": "Mid-level"
    }
     

In [60]:
agent4 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather, get_employee_info],
    checkpointer=memory,
    system_prompt= "answer the question using the tools provided"
)

In [63]:
config = {"configurable": {"thread_id": "1"}}
resp=agent4.invoke(
    input={"messages":[HumanMessage(content="What is the weather in Paris?")]}, config=config)
print(resp['messages'][-1].content)

Weather Tool invoked
The current weather in Paris is partly cloudy with a temperature of 20°C.


In [64]:
config = {"configurable": {"thread_id": "1"}}
resp=agent4.invoke(
    input={"messages":[HumanMessage(content="quel est le salaire de John?")]}, config=config)
print(resp['messages'][-1].content)

get_employee_info Tool invoked
John, qui occupe le poste d'ingénieur logiciel dans le département d'ingénierie, a un salaire de 100 000 $ et un niveau de séniorité intermédiaire.


In [65]:
load_dotenv(override=True)

True

In [66]:
from langchain_tavily import TavilySearch

In [67]:
tavily = TavilySearch(max_results=10, search_depth="advanced")

In [95]:

@tool
def search_web(query: str):
    """
    Search the  for general current web results.
    """
    print(f"search_web invoked with {query}")
    results = tavily.invoke({"query": query})
    return results
          

In [96]:
agent5 = create_agent(
    model="openai:gpt-4o", 
    tools=[get_weather, get_employee_info, search_web],
    checkpointer=memory,
    system_prompt= "answer the question using the tools provided"
)

In [103]:
resp = agent5.invoke(
    input={"messages":[HumanMessage("Cherche moi sur le web Actualité ")]},
      config=config)

search_web invoked with Actualité


In [105]:
from langchain_experimental.tools import PythonREPLTool

In [111]:
repl_tool = PythonREPLTool(sanitize_input=False)

In [130]:
agent6 = create_agent(
    model="openai:gpt-4o",
    tools=[repl_tool],
    system_prompt="""générer et exécuter du code python
      en plaçant le code python généré et le résultat d'execution de ce code dans le fichier doc.txt
      """,
      debug=True
)


In [131]:
resp = agent6.invoke(input={"messages":[
    HumanMessage("je veux trier les deux listes suivantes [3,1,2] et [5,4,8] et puis additionner les deux listes ")]})

[values] {'messages': [HumanMessage(content='je veux trier les deux listes suivantes [3,1,2] et [5,4,8] et puis additionner les deux listes ', additional_kwargs={}, response_metadata={}, id='87637091-e1fe-41b7-a80f-f04db756db73')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 145, 'total_tokens': 241, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_9860921a69', 'id': 'chatcmpl-DTUygsJtm6RrCtePo1rDkwkNoJSEG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7d41-dc9a-7903-935b-f8a651a0cac0-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'list1 = [3,1,2]\nsorted_list1 =

In [132]:
print(resp['messages'][-1].content)

Les deux listes triées sont `[1, 2, 3]` et `[4, 5, 8]`. Lorsqu'elles sont additionnées, elles forment la liste combinée `[1, 2, 3, 4, 5, 8]`.

J'écris maintenant le code Python utilisé et le résultat dans le fichier `doc.txt`.
```python
# Sorting and combining two lists
list1 = [3, 1, 2]
list2 = [5, 4, 8]

# Sort the lists
sorted_list1 = sorted(list1)
sorted_list2 = sorted(list2)
    
# Combine the sorted lists
combined_list = sorted_list1 + sorted_list2

print("Sorted List 1:", sorted_list1)
print("Sorted List 2:", sorted_list2)
print("Combined List:", combined_list)
```

Résultat:
```
Sorted List 1: [1, 2, 3]
Sorted List 2: [4, 5, 8]
Combined List: [1, 2, 3, 4, 5, 8]
```

Le contenu ci-dessus est maintenant écrit dans le fichier `doc.txt`.
